In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week9-lesson-3"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

## Bucketing -- saveasTable

In [5]:
spark.sql("create database itv027484")

""


In [2]:
cust_df = spark.read.format('csv').option('inferSchema','true').load('/public/trendytech/retail_db/customers/part-00000')

In [3]:
cust_df1 = cust_df.toDF("cust_id","cust_fname","cust_lname","cust_email","cust_pass","cust_addr1","cust_city","cust_state","cust_zip")

In [ ]:
cust_df1.write \
.mode('overwrite') \
.format('parquet') \
.bucketBy(8,"cust_id") \
.saveAsTable("itv027484.cust_bkt")

In [ ]:
# [itv027484@g02 ~]$ hadoop fs -ls /user/itv027484/warehouse/itv027484.db/cust_bkt
# Found 9 items
# -rw-r--r--   3 itv027484 supergroup          0 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/_SUCCESS
# -rw-r--r--   3 itv027484 supergroup      50482 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/part-00000-47c86da9-4a79-459d-ae3c-0ceb098bf1ec_00000.c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup      50151 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/part-00000-47c86da9-4a79-459d-ae3c-0ceb098bf1ec_00001.c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup      50581 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/part-00000-47c86da9-4a79-459d-ae3c-0ceb098bf1ec_00002.c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup      50267 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/part-00000-47c86da9-4a79-459d-ae3c-0ceb098bf1ec_00003.c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup      50021 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/part-00000-47c86da9-4a79-459d-ae3c-0ceb098bf1ec_00004.c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup      51855 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/part-00000-47c86da9-4a79-459d-ae3c-0ceb098bf1ec_00005.c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup      51680 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/part-00000-47c86da9-4a79-459d-ae3c-0ceb098bf1ec_00006.c000.snappy.parquet
# -rw-r--r--   3 itv027484 supergroup      49932 2026-08-05 11:28 /user/itv027484/warehouse/itv027484.db/cust_bkt/part-00000-47c86da9-4a79-459d-ae3c-0ceb098bf1ec_00007.c000.snappy.parquet

In [18]:
spark.sql("select * from itv027484.cust_bkt where cust_id = 10").show()

+-------+----------+----------+----------+---------+--------------------+---------+----------+--------+
|cust_id|cust_fname|cust_lname|cust_email|cust_pass|          cust_addr1|cust_city|cust_state|cust_zip|
+-------+----------+----------+----------+---------+--------------------+---------+----------+--------+
|     10|   Melissa|     Smith| XXXXXXXXX|XXXXXXXXX|8598 Harvest Beac...| Stafford|        VA|   22554|
+-------+----------+----------+----------+---------+--------------------+---------+----------+--------+



In [19]:
cust_df1.filter("cust_id = 10").show()

+-------+----------+----------+----------+---------+--------------------+---------+----------+--------+
|cust_id|cust_fname|cust_lname|cust_email|cust_pass|          cust_addr1|cust_city|cust_state|cust_zip|
+-------+----------+----------+----------+---------+--------------------+---------+----------+--------+
|     10|   Melissa|     Smith| XXXXXXXXX|XXXXXXXXX|8598 Harvest Beac...| Stafford|        VA|   22554|
+-------+----------+----------+----------+---------+--------------------+---------+----------+--------+



#### savings due to bucketing might not be visible in all versions of spark